# Haystack

**Module:** 09-llm-frameworks

**Notebook:** `04-haystack.ipynb`

This expanded lesson goes beyond definitions: each topic includes *why it matters*, *how it works*, intuition, pitfalls, and when to use it—plus runnable Python demos, comparison aids, and exercises.


## Learning Objectives

By the end of this notebook, you will be able to:

- Explain and apply **Haystack Overview** with clear contracts and failure modes
- Explain and apply **Document Stores** with clear contracts and failure modes
- Explain and apply **Pipeline Graph** with clear contracts and failure modes
- Explain and apply **RAG with Haystack Concepts** with clear contracts and failure modes
- Explain and apply **When to Choose Haystack** with clear contracts and failure modes
- Explain and apply **Self-Check** with clear contracts and failure modes
- Evaluate tradeoffs (quality, cost, latency, safety) for designs in this lesson
- Implement small Python prototypes that make the ideas testable


## How to Use This Notebook

1. Read the topic sections fully—do not jump only to code.
2. Run each demo; then change inputs to break them and fix them.
3. API examples use placeholders like `YOUR_API_KEY` or `os.environ.get(...)`.
4. Keep secrets out of git; treat prompts/tool schemas as versioned code.
5. Complete the **Try It Yourself** exercises before moving on.


### Pipeline walkthrough — Haystack

```mermaid
flowchart LR
  A[Problem / user goal] --> B[Contract: IO + constraints]
  B --> C[Implement core path]
  C --> D[Validate / guardrails]
  D --> E[Eval fixtures]
  E --> F[Observe in production]
  F -->|regressions| B
```

```text
goal -> contract -> implement -> validate -> evaluate -> monitor -> revise
```


## Curriculum Map

This notebook's spine (preserve/cover all of these):

1. **Haystack Overview**
2. **Document Stores**
3. **Pipeline Graph**
4. **RAG with Haystack Concepts**
5. **When to Choose Haystack**
6. **Self-Check**

Read top-to-bottom once, then revisit weak spots with the exercises.


## Haystack Overview

### Definition
**Haystack Overview** is a core building block in 04-haystack within LLM frameworks. Treat it as an orchestration layer—useful only when it clarifies ownership of steps: something you can name, version, test, and operate.

### Why it matters
In LLM frameworks, weak designs around Haystack Overview typically surface as framework lock-in, opaque magic, and untested compositions. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Haystack Overview: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like runnable chains, indexes, and portable pipelines.

### Intuition
Explain Haystack Overview as an orchestration layer—useful only when it clarifies ownership of steps. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Haystack Overview as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Haystack Overview
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM frameworks: framework lock-in, opaque magic, and untested compositions

### When to use
Use Haystack Overview when your product path depends on this concern in LLM frameworks. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.

### Quick reference

| Lens | Question |
|------|----------|
| Product | What user outcome does Haystack Overview improve? |
| Engineering | What is the interface / data contract? |
| Safety | What can go wrong if it fails open? |
| Ops | How will we notice regressions? |


In [ ]:
# Demo: make "Haystack Overview" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Haystack Overview"
    notebook: str = "04-haystack"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_0 = ConceptContract()
print(json.dumps({"contract": asdict(contract_0), "health": contract_0.health()}, indent=2))


In [ ]:
# Framework mental model: compose runnable steps
class Runnable:
    def __init__(self, fn): self.fn = fn
    def __or__(self, other):
        return Runnable(lambda x: other.fn(self.fn(x)))
    def invoke(self, x): return self.fn(x)

prompt = Runnable(lambda q: f"Answer briefly: {q}")
model = Runnable(lambda p: f"<model-output for: {p[:40]}...>")
parse = Runnable(lambda t: {"text": t, "n": len(t)})
chain = prompt | model | parse
print(chain.invoke("What is LCEL?"))


In [ ]:
# Index → retrieve → generate sketch (RAG-shaped)
NODES = [{"id": 1, "text": "LCEL composes runnables with |"}, {"id": 2, "text": "LlamaIndex focuses on data indexes"}]

def retrieve(q, k=1):
    return sorted(NODES, key=lambda n: len(set(q.lower().split()) & set(n["text"].lower().split())), reverse=True)[:k]

def generate(q, nodes):
    ctx = " | ".join(n["text"] for n in nodes)
    return f"Q: {q}\nCTX: {ctx}\nA: Based on context, {nodes[0]['text']}."

print(generate("What is LCEL?", retrieve("LCEL compose")))


In [ ]:
# Demo: decision table for applying "Haystack Overview"
options = [
    {"option": "baseline_simple", "quality": 0.7, "cost": 1, "ops": 0.9},
    {"option": "advanced_haystack_ove", "quality": 0.85, "cost": 3, "ops": 0.6},
]
for o in options:
    o["utility"] = round(o["quality"] * 2 - 0.3*o["cost"] + 0.5*o["ops"], 3)
best = max(options, key=lambda x: x["utility"])
print("ranked:", sorted(options, key=lambda x: -x["utility"]))
print("prefer:", best["option"])


## Document Stores

### Definition
**Document Stores** is a core building block in 04-haystack within LLM frameworks. Treat it as an orchestration layer—useful only when it clarifies ownership of steps: something you can name, version, test, and operate.

### Why it matters
In LLM frameworks, weak designs around Document Stores typically surface as framework lock-in, opaque magic, and untested compositions. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Document Stores: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like runnable chains, indexes, and portable pipelines.

### Intuition
Explain Document Stores as an orchestration layer—useful only when it clarifies ownership of steps. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Document Stores as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Document Stores
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM frameworks: framework lock-in, opaque magic, and untested compositions

### When to use
Use Document Stores when your product path depends on this concern in LLM frameworks. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Document Stores" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Document Stores"
    notebook: str = "04-haystack"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_1 = ConceptContract()
print(json.dumps({"contract": asdict(contract_1), "health": contract_1.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Document Stores"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Document Stores"}
strong = {"definition": "Document Stores", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Document Stores"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Document Stores", "passed": len(checks)-len(failed), "failed": failed})


### Worked scenario — Document Stores

**Situation:** A team wants to productionize a feature involving **Document Stores**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Pipeline Graph

### Definition
**Pipeline Graph** is a core building block in 04-haystack within LLM frameworks. Treat it as an orchestration layer—useful only when it clarifies ownership of steps: something you can name, version, test, and operate.

### Why it matters
In LLM frameworks, weak designs around Pipeline Graph typically surface as framework lock-in, opaque magic, and untested compositions. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Pipeline Graph: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like runnable chains, indexes, and portable pipelines.

### Intuition
Explain Pipeline Graph as an orchestration layer—useful only when it clarifies ownership of steps. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Pipeline Graph as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Pipeline Graph
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM frameworks: framework lock-in, opaque magic, and untested compositions

### When to use
Use Pipeline Graph when your product path depends on this concern in LLM frameworks. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Pipeline Graph" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Pipeline Graph"
    notebook: str = "04-haystack"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_2 = ConceptContract()
print(json.dumps({"contract": asdict(contract_2), "health": contract_2.health()}, indent=2))


In [ ]:
class Pipeline:
    def __init__(self):
        self.stages = []
    def add(self, name, fn):
        self.stages.append((name, fn)); return self
    def run(self, data):
        audit = []
        for name, fn in self.stages:
            data = fn(data)
            audit.append({"stage": name, "type": type(data).__name__})
        return data, audit

result, audit = (
    Pipeline()
    .add("normalize", lambda s: s.strip().lower())
    .add("tokens", lambda s: s.split())
    .add("features", lambda toks: {"n": len(toks), "head": toks[:3]})
    .run("  SSO Login Loop  ")
)
print(result); print(audit)


In [ ]:
# Parallel fan-out / fan-in sketch
from concurrent.futures import ThreadPoolExecutor

def analyze(kind, text):
    return {"kind": kind, "len": len(text)}

text = "investigate checkout latency"
with ThreadPoolExecutor(max_workers=3) as ex:
    parts = list(ex.map(lambda k: analyze(k, text), ["security", "perf", "ux"]))
merged = {p["kind"]: p["len"] for p in parts}
print(merged)


## RAG with Haystack Concepts

### Definition
**RAG with Haystack Concepts** is a core building block in 04-haystack within LLM frameworks. Treat it as an orchestration layer—useful only when it clarifies ownership of steps: something you can name, version, test, and operate.

### Why it matters
In LLM frameworks, weak designs around RAG with Haystack Concepts typically surface as framework lock-in, opaque magic, and untested compositions. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For RAG with Haystack Concepts: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like runnable chains, indexes, and portable pipelines.

### Intuition
Explain RAG with Haystack Concepts as an orchestration layer—useful only when it clarifies ownership of steps. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating RAG with Haystack Concepts as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for RAG with Haystack Concepts
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM frameworks: framework lock-in, opaque magic, and untested compositions

### When to use
Use RAG with Haystack Concepts when your product path depends on this concern in LLM frameworks. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "RAG with Haystack Concepts" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "RAG with Haystack Concepts"
    notebook: str = "04-haystack"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_3 = ConceptContract()
print(json.dumps({"contract": asdict(contract_3), "health": contract_3.health()}, indent=2))


In [ ]:
# Framework mental model: compose runnable steps
class Runnable:
    def __init__(self, fn): self.fn = fn
    def __or__(self, other):
        return Runnable(lambda x: other.fn(self.fn(x)))
    def invoke(self, x): return self.fn(x)

prompt = Runnable(lambda q: f"Answer briefly: {q}")
model = Runnable(lambda p: f"<model-output for: {p[:40]}...>")
parse = Runnable(lambda t: {"text": t, "n": len(t)})
chain = prompt | model | parse
print(chain.invoke("What is LCEL?"))


In [ ]:
# Index → retrieve → generate sketch (RAG-shaped)
NODES = [{"id": 1, "text": "LCEL composes runnables with |"}, {"id": 2, "text": "LlamaIndex focuses on data indexes"}]

def retrieve(q, k=1):
    return sorted(NODES, key=lambda n: len(set(q.lower().split()) & set(n["text"].lower().split())), reverse=True)[:k]

def generate(q, nodes):
    ctx = " | ".join(n["text"] for n in nodes)
    return f"Q: {q}\nCTX: {ctx}\nA: Based on context, {nodes[0]['text']}."

print(generate("What is LCEL?", retrieve("LCEL compose")))


### Worked scenario — RAG with Haystack Concepts

**Situation:** A team wants to productionize a feature involving **RAG with Haystack Concepts**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## When to Choose Haystack

### Definition
**When to Choose Haystack** helps you choose among alternatives using explicit criteria rather than hype.

### Why it matters
In LLM frameworks, weak designs around When to Choose Haystack typically surface as framework lock-in, opaque magic, and untested compositions. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
List options, define criteria (quality, cost, latency, ops, lock-in), score with evidence, document the decision.

### Intuition
Explain When to Choose Haystack as an orchestration layer—useful only when it clarifies ownership of steps. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating When to Choose Haystack as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for When to Choose Haystack
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM frameworks: framework lock-in, opaque magic, and untested compositions

### When to use
Use When to Choose Haystack when your product path depends on this concern in LLM frameworks. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "When to Choose Haystack" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "When to Choose Haystack"
    notebook: str = "04-haystack"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_4 = ConceptContract()
print(json.dumps({"contract": asdict(contract_4), "health": contract_4.health()}, indent=2))


In [ ]:
# Framework mental model: compose runnable steps
class Runnable:
    def __init__(self, fn): self.fn = fn
    def __or__(self, other):
        return Runnable(lambda x: other.fn(self.fn(x)))
    def invoke(self, x): return self.fn(x)

prompt = Runnable(lambda q: f"Answer briefly: {q}")
model = Runnable(lambda p: f"<model-output for: {p[:40]}...>")
parse = Runnable(lambda t: {"text": t, "n": len(t)})
chain = prompt | model | parse
print(chain.invoke("What is LCEL?"))


In [ ]:
# Index → retrieve → generate sketch (RAG-shaped)
NODES = [{"id": 1, "text": "LCEL composes runnables with |"}, {"id": 2, "text": "LlamaIndex focuses on data indexes"}]

def retrieve(q, k=1):
    return sorted(NODES, key=lambda n: len(set(q.lower().split()) & set(n["text"].lower().split())), reverse=True)[:k]

def generate(q, nodes):
    ctx = " | ".join(n["text"] for n in nodes)
    return f"Q: {q}\nCTX: {ctx}\nA: Based on context, {nodes[0]['text']}."

print(generate("What is LCEL?", retrieve("LCEL compose")))


## Self-Check

### Definition
**Self-Check** is a core building block in 04-haystack within LLM frameworks. Treat it as an orchestration layer—useful only when it clarifies ownership of steps: something you can name, version, test, and operate.

### Why it matters
In LLM frameworks, weak designs around Self-Check typically surface as framework lock-in, opaque magic, and untested compositions. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Self-Check: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like runnable chains, indexes, and portable pipelines.

### Intuition
Explain Self-Check as an orchestration layer—useful only when it clarifies ownership of steps. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Self-Check as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Self-Check
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM frameworks: framework lock-in, opaque magic, and untested compositions

### When to use
Use Self-Check when your product path depends on this concern in LLM frameworks. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Self-Check" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Self-Check"
    notebook: str = "04-haystack"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_5 = ConceptContract()
print(json.dumps({"contract": asdict(contract_5), "health": contract_5.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Self-Check"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Self-Check"}
strong = {"definition": "Self-Check", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Self-Check"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Self-Check", "passed": len(checks)-len(failed), "failed": failed})


### Worked scenario — Self-Check

**Situation:** A team wants to productionize a feature involving **Self-Check**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Comparison Snapshot

Use this table when reviewing designs in **Haystack**.

| Topic | Do | Don't |
|-------|----|-------|
| Haystack Overview | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Document Stores | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Pipeline Graph | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| RAG with Haystack Concepts | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| When to Choose Haystack | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Self-Check | Design carefully; measure; bound cost | Skipping eval / unbounded loops |


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| Haystack Overview | Key concept covered in this notebook; see its section for definition and pitfalls |
| Document Stores | Key concept covered in this notebook; see its section for definition and pitfalls |
| Pipeline Graph | Key concept covered in this notebook; see its section for definition and pitfalls |
| RAG with Haystack Concepts | Key concept covered in this notebook; see its section for definition and pitfalls |
| When to Choose Haystack | Key concept covered in this notebook; see its section for definition and pitfalls |
| Self-Check | Key concept covered in this notebook; see its section for definition and pitfalls |


## Summary & Key Takeaways

- **Haystack** is a production concern: contracts, evals, and guardrails beat vibe-driven prompting.
- Every major topic above includes definition, motivation, mechanism, intuition, pitfalls, and usage guidance—use that checklist in design reviews.
- Prefer small, measurable demos before framework sprawl.
- Bound loops, validate tool args, and keep API keys in environment variables (`YOUR_API_KEY` is a placeholder only).
- Carry forward: connect these ideas to the next notebooks in **09-llm-frameworks**.


## Try It Yourself

1. Implement a failing test/fixture for **Haystack Overview**, then fix your demo until it passes.
2. Implement a failing test/fixture for **Document Stores**, then fix your demo until it passes.
3. Implement a failing test/fixture for **Pipeline Graph**, then fix your demo until it passes.
4. Implement a failing test/fixture for **RAG with Haystack Concepts**, then fix your demo until it passes.
5. Implement a failing test/fixture for **When to Choose Haystack**, then fix your demo until it passes.
6. Estimate token cost for your prompt/tool trace at 1k and 100k requests/day.
7. Write a 5-row comparison of two design alternatives from this notebook; pick one with explicit criteria.
8. Red-team your solution with empty input, hostile input, and a tool/API timeout.
